# Introduction

Introduce the project, and  your approach, talk about the process of how you came up with the metric and some alternatives you may have explored.

# The Metric

Describe your metric, and what features you are measuring. What datasets are you using?

# The Best Neighborhood

## Import Packages and Data Files

In [10]:
# import necessary packages
import numpy as np
import pandas as pd
import plotly.express as px
import json

# import geojson data
with open("neighborhoods.geojson", "r", encoding="utf-8") as f:
    pgh_nbs = json.load(f)

# import neighborhood data
nb = pd.read_csv("neighborhoods.csv")

# crate list of all neighborhood names
names = nb['hood'].unique()
names.sort()

# create initial metric_df from nb data
metric_df = nb[["hood","sqmiles","intptlat10","intptlon10"]]
metric_df.rename(columns={"hood":"neighborhood","intptlat10":"lat","intptlon10":"lon"}, inplace=True)

# import facilities data
fac_df = pd.read_csv("facilities.csv")


/tmp/ipykernel_3123/1583327946.py:23: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## Process Facilities Dataset

In [11]:
# filter results to only have official neighborhoods (removes 1 row)
fac_df = fac_df.loc[fac_df["neighborhood"].isin(names)]

# filter results to remove rows where "inactive" value isn't "f"
fac_df = fac_df.loc[fac_df["inactive"] == "f"]

# change "type" strings to all have title case
fac_df["type"] = fac_df["type"].str.title()

# make list of all different facility types
type_list = list(fac_df["type"].unique())
# make list of facility values
value_list = [0,1,5,5,0,5,1,0,1,0.1,10,5,0,0,0,5,0,1,0,0,0,0,1]

# combine types and values into dataframe to merge with fac_df
value_df = pd.DataFrame({"type": type_list, "value": value_list})

# merge with fac_df
fac_df = pd.merge(fac_df, value_df, how="left", on="type")

# remove duplicate pool buildings
df_pools = fac_df.loc[fac_df.type=="Pool"].drop_duplicates(subset="parcel_id")
df_not_pools = fac_df.loc[fac_df.type!="Pool"]
fac_df = pd.concat([df_pools, df_not_pools])

# total value of facilites per neighborhood
fac_df = fac_df.groupby('neighborhood')['value'].sum()
fac_df = fac_df.reset_index()

# merge into metric_df
metric_df = pd.merge(metric_df, fac_df, on="neighborhood", how="outer")

# assign value of 0 to neighborhoods without any facilities
metric_df.loc[metric_df['value'].isna(),'value']=0

# rename "value" col and create new col adjusted for area
metric_df.rename(columns={"value":"fac_score"}, inplace=True)
metric_df["adj_fac_score"] = metric_df["fac_score"] / metric_df["sqmiles"]

## Process Crime Dataset

## Process Trees Dataset

## Combine Datasets

In [12]:
metric_df.describe()

,sqmiles,fac_score,adj_fac_score
count,90.000000,90.000000,90.000000
mean,0.615622,5.498889,8.981846
std,0.481316,7.149927,12.487572
min,0.103298,0.000000,0.000000
25%,0.284284,0.000000,0.000000
50%,0.443539,2.500000,4.850624
75%,0.824866,8.775000,12.963862
max,2.676605,33.000000,58.728084


## Plot Individual Metrics

In [13]:
# create scatter plot showing value per sqmile
fig1 = px.scatter_map(metric_df,
                      size='adj_fac_score',
                      lat="lat",lon="lon",
                      zoom=10, center={"lat":40.4387, "lon":-79.9972},
                      map_style="streets",
                      title = "Neighborhood Facility Value")
fig1.update_layout(margin={"r":0,"l":0,"b":0},
                   title={"xanchor":"center", "x":0.5})
fig1.show()

## Plot Combined Metric

## Determine Best Neighborhood

# Conclusion

Reflect on how the data-driving determination of "best neighborhood" is the same or different from your personal favorite neighborhood. Each member of the group should write their own response to this.